# 06. Threshold and entity merging

A token classifier gives probabilities for O, B-LOC and I-LOC per token; turning them into place strings
takes three decisions (`ner/predict.py`):

- **threshold**: a token is part of a place when P(B) + P(I) is at least this value. 0.5 is the plain
  argmax over "place / not a place"; lower values trade precision for recall.
- **aggregation**: decide per token, or once per word from the first token, the mean or the max of its
  tokens. Word-level decisions never cut a word in half ("Київ" + "щині").
- **merging**: `bio` starts a new place at a word that looks like a beginning (P(B) > P(I)); `io` joins
  all neighbouring place tokens. They differ on lists like "Києві, Львові" vs "вулиці Хрещатик".

Nothing is retrained: we decode the probabilities saved by the training runs. To keep the final score
honest we **tune on half A** (test posts the generator saw, but no model trained on) and **report on
half B and the gold posts**. Tuning on the training sentences would pick the sentence labeling style.

Runs on CPU in a minute or two. It needs the run folders (with `probs.pkl`) of the chosen model.

### Setup

On Colab (T4 or better): clones the repo, installs W&B and keeps every run on Drive in
`dl-a4-ner/runs`, so a disconnect loses at most the run in progress (finished runs are loaded, not
retrained). Teammates: put a shortcut to the shared `dl-a4-ner` folder in your My Drive, so all runs
land in one place. Set `WANDB_ENTITY` to the team's W&B entity to log into one shared project.
Locally the notebook runs from the repo root and writes to `transformers/runs`. To check the whole
notebook in a few minutes before using GPU time, set the environment variable `NER_SMOKE=1`: every run
shrinks to a few hundred texts and goes to `transformers/runs_smoke`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/koalla05/deep-learning-assignment4.git"
WANDB_PROJECT = "dl-a4-ner"  # None switches W&B off
WANDB_ENTITY = None  # the team's W&B entity, so all teammates log into one project

if "google.colab" in sys.modules:
    repo = Path("/content/deep-learning-assignment4")
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/dl-a4-ner/runs")
else:
    if Path.cwd().name == "transformers":
        os.chdir("..")
    OUT = Path("transformers/runs")
sys.path.insert(0, os.getcwd())
if os.environ.get("NER_SMOKE"):  # quick end-to-end check with tiny runs, see ner.train.smoke
    OUT, WANDB_PROJECT = Path("transformers/runs_smoke"), None
OUT.mkdir(parents=True, exist_ok=True)
if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
if WANDB_PROJECT:
    import wandb

    wandb.login()

In [ ]:
import itertools

import pandas as pd

from ner.cv import fold_average
from ner.data import load_test_posts
from ner.evaluate import eval_sets, score, submission
from ner.predict import decode_all

# the three fold runs of the model to post-process (the best model notebook's FOLD_RUNS)
FOLD_RUNS = ["03-xlmr-large-lr1e-05-fold0", "03-xlmr-large-lr1e-05-fold1", "03-xlmr-large-lr1e-05-fold2"]
probs, texts = fold_average(OUT, FOLD_RUNS)
truth = {k: v[1] for k, v in eval_sets(None).items()}


def evaluate(**kw):
    return {s: score(truth[s], decode_all(texts[s], probs[s], **kw)) for s in ("half_A", "half_B", "gold26")}

## Grid on half A

In [ ]:
GRID = {
    "threshold": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7],
    "aggregation": ["token", "first", "mean", "max"],
    "merge": ["bio", "io"],
}
rows = []
for values in itertools.product(*GRID.values()):
    kw = dict(zip(GRID, values))
    m = evaluate(**kw)
    rows.append({**kw, **{f"{s} {k}": v[k] for s, v in m.items() for k in ("f1", "precision", "recall")}})
grid = pd.DataFrame(rows)
grid.sort_values("half_A f1", ascending=False).head(10).round(4)

In [ ]:
default = grid.query("threshold == 0.5 and aggregation == 'token' and merge == 'bio'")
best = grid.loc[grid["half_A f1"].idxmax()]
pd.DataFrame({"default (0.5, token, bio)": default.iloc[0], "best on half A": best}).loc[
    ["threshold", "aggregation", "merge", "half_A f1", "half_B f1", "half_B precision", "half_B recall", "gold26 f1"]
]

How F1 on each set moves with the threshold (best aggregation and merging held fixed):

In [ ]:
fixed = grid[(grid.aggregation == best.aggregation) & (grid.merge == best.merge)].set_index("threshold")
fixed[["half_A f1", "half_B f1", "gold26 f1"]].plot(marker="o", figsize=(6, 3.2), title="F1 vs threshold")

In [ ]:
grid.groupby(["aggregation", "merge"])[["half_A f1", "half_B f1"]].max().round(4)

## Submission with the tuned decoding

In [ ]:
DECODE = {k: best[k] for k in GRID}
test = load_test_posts()
by_id = {}
for half in "AB":
    by_id.update(zip(test[test.half == half].text_id, decode_all(texts[f"half_{half}"], probs[f"half_{half}"], **DECODE)))
sub = submission(list(test.text_id), [by_id[t] for t in test.text_id])
path = OUT / f"{FOLD_RUNS[0].rsplit('-fold', 1)[0]}-tuned-submission.csv"
sub.to_csv(path, index=False)
print(DECODE, "->", path)

## Conclusions

- Does the threshold tuned on half A also help on half B and the gold posts? If it helps only on half A,
  it fit the noise of 239 posts.
- Which aggregation and merging rule wins, and on what kind of error (precision or recall)?
- Compare the leaderboard score of this submission with the untuned one from the model notebook.